# LangGraph Hands-On: From First Graph to Production Agent

> **Who is this for:** learners who know what an agent is (an LLM that calls tools in a loop) and have read the short [LangGraph overview](03_langgraph.ipynb). Now you want to *build* with LangGraph, one feature at a time, and understand what each piece is for.
>
> **What you will learn:**
> - **why graphs**: how a graph differs from a fixed chain and from a free-form `while` loop
> - the core building blocks: `StateGraph`, **state** (`TypedDict`), **reducers** (`operator.add`, `add_messages`), **nodes**, **edges**, `START` / `END`
> - **conditional edges** (routing), **cycles** with a stop condition, `compile`, `invoke` and `stream` (modes `values`, `updates`, `messages`, `custom`)
> - drawing the graph as **Mermaid** text
> - a **tool-calling agent** built by hand with `ToolNode` + `tools_condition`, and the prebuilt `create_react_agent`
> - **checkpointers** and `thread_id` (conversation memory), **time travel** (history, replay, fork)
> - **human-in-the-loop** with `interrupt()` and `Command(resume=...)`, and `Command(goto=..., update=...)`
> - the **Send API** for parallel map-reduce, **subgraphs**, and the long-term memory **Store**
> - real, runnable **multi-agent patterns**: supervisor, handoffs (swarm-style), hierarchical teams, plan-and-execute, reflection
> - production concerns: error handling, `RetryPolicy`, `recursion_limit`, token streaming to a UI, deployment and LangSmith tracing
>
> **Prerequisites:** [Agent fundamentals](01_agent_fundamentals.ipynb), [Memory and planning](02_memory_and_planning.ipynb), [LangGraph overview](03_langgraph.ipynb). Helpful: [MCP and tools](05_mcp_and_tools.ipynb), [Agent observability](06_agent_observability.ipynb). Companion notebooks: [Harness engineering and agentic engineering](10_harness_engineering_and_agentic_engineering.ipynb), [Context engineering, memory and agent protocols](11_context_engineering_memory_and_agent_protocols.ipynb), [CrewAI and other multi-agent frameworks, hands-on](13_crewai_and_multi_agent_frameworks_hands_on.ipynb).
>
> **Time needed:** about 2-3 hours to read and play (the code itself runs in a few seconds).
>
> **Runs on:** CPU, no API key, no internet. Every "LLM" here is a **stand-in**: either a plain Python function with scripted answers, or LangChain's `GenericFakeChatModel`, which replays messages we give it. We say so each time. Tested with langgraph 1.x and langchain-core 1.x (as of late 2026).

## 0. Setup

We import the standard library, and the two LangGraph-related packages:

- **langgraph**: the graph runtime (state, nodes, edges, checkpoints).
- **langchain-core**: message classes (`HumanMessage`, `AIMessage`, ...), the `@tool` decorator and the fake chat models.

We also define one helper, `ScriptedChatModel`. It is a fake LLM: it returns, in order, the replies we put in a list. Real chat models have a `bind_tools()` method (it tells the model which tools exist); our fake one just ignores the tools and keeps replying from its script.

In [1]:
import operator, random, time, json, os, warnings, uuid
from typing import Annotated, Literal
from typing_extensions import TypedDict

import langgraph, langchain_core
from importlib.metadata import version
from langchain_core.messages import (HumanMessage, AIMessage, ToolMessage,
                                     SystemMessage, RemoveMessage, AnyMessage)
from langchain_core.tools import tool
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, Send, interrupt, RetryPolicy
from langgraph.store.memory import InMemoryStore

random.seed(42)
print("langgraph", version("langgraph"), "| langchain-core", version("langchain-core"))

class ScriptedChatModel(GenericFakeChatModel):
    """A FAKE chat model (stand-in for a real LLM). It replays the AIMessages
    (or strings) we pass in `messages=iter([...])`, one per call."""
    def bind_tools(self, tools, **kwargs):
        return self          # a real model would remember the tool schemas here

def show(messages):
    """Print a list of chat messages in a short, readable way."""
    for m in messages:
        extra = f"  tool_calls={[(t['name'], t['args']) for t in m.tool_calls]}" if getattr(m, "tool_calls", None) else ""
        print(f"  {type(m).__name__:<13} {str(m.content)[:70]!r}{extra}")

langgraph 1.2.21 | langchain-core 1.8.0


## 1. Why graphs? Chains vs free-form loops vs graphs

**What it is.** LangGraph lets you write an LLM application as a **graph**: boxes (**nodes**, which are Python functions) joined by arrows (**edges**, which say what runs next). All nodes read and write one shared object called the **state**.

**Simple analogy.** Think of a board game. The **board** (squares and arrows) is the graph. Your **game piece and score card** are the state. Each square tells you what to do, then the arrows tell you where to move. Some arrows depend on a dice roll (conditional edges), and some lead you back to an earlier square (cycles).

There are three common ways to wire an LLM app:

| Style | What it looks like | Good at | Weak at |
|---|---|---|---|
| **Chain** (pipeline) | step 1 -> step 2 -> step 3, always in that order | simple, predictable tasks (summarise, then translate) | no branches, no loops, no "try again" |
| **Free-form agent loop** | `while True: ask LLM what to do; do it` | open-ended tasks | hard to debug, hard to pause, easy to loop forever, the control flow is hidden inside prompts |
| **Graph** (LangGraph) | named steps + explicit arrows, with branches and loops | agents that need loops **and** control: retries, approvals, memory, parallel work | a bit more code up front |

```mermaid
flowchart LR
    subgraph Chain
      a1[step 1] --> a2[step 2] --> a3[step 3]
    end
    subgraph Graph
      b0([START]) --> b1[agent]
      b1 -- needs a tool --> b2[tools]
      b2 --> b1
      b1 -- done --> b3([END])
    end
```

**How it works (the LangGraph runtime in 4 steps).**
1. You describe the **state** shape (which fields exist).
2. You add **nodes**. Each node gets the current state and returns a **partial update** (only the fields it changes).
3. You add **edges** (fixed or conditional).
4. You **compile** the graph and run it. The runtime runs nodes in "super-steps" (rounds), merges their updates into the state, and can save a **checkpoint** (a snapshot of the state) after every round.

**When to use it / when not.** Use LangGraph when you need loops, branches, pausing for a human, memory across turns, or several agents. For a one-shot prompt or a fixed 2-step pipeline, a plain function is simpler.

## 2. Your first graph: state, nodes, edges, START and END

**What it is.**
- **State**: a `TypedDict` (a Python dict with named, typed keys) that every node can read.
- **Node**: a function `state -> dict`. The returned dict is a **partial update**, not the whole state.
- **Edge**: `add_edge("a", "b")` means "after `a` finishes, run `b`".
- **START** and **END**: two special built-in nodes for "where the run begins" and "where it stops".

**Analogy.** A relay race: each runner (node) takes the baton (state), adds something, and hands it to the next runner along a fixed track (edges).

In [2]:
class GreetState(TypedDict):
    name: str
    greeting: str
    shout: str

def make_greeting(state: GreetState) -> dict:
    # A node returns ONLY the keys it wants to change.
    return {"greeting": f"Hello, {state['name']}!"}

def make_shout(state: GreetState) -> dict:
    return {"shout": state["greeting"].upper()}

builder = StateGraph(GreetState)          # 1. describe the state
builder.add_node("greet", make_greeting)  # 2. add nodes (name -> function)
builder.add_node("shout", make_shout)
builder.add_edge(START, "greet")          # 3. add edges
builder.add_edge("greet", "shout")
builder.add_edge("shout", END)

graph = builder.compile()                 # 4. compile (checks the wiring)
result = graph.invoke({"name": "Ada"})    # 5. run it with a starting state
print(result)

{'name': 'Ada', 'greeting': 'Hello, Ada!', 'shout': 'HELLO, ADA!'}


**What just happened?** `compile()` turned the builder into a runnable object (a `CompiledStateGraph`). `invoke()` ran `greet`, merged `{"greeting": ...}` into the state, ran `shout`, merged again, then reached `END` and returned the final state.

Two things to notice:
- `make_shout` never had to pass `name` along. The runtime keeps every key; nodes only send what changed.
- `compile()` checks for mistakes early, for example an edge pointing at a node that does not exist.

## 3. Reducers: how updates are merged into the state

**What it is.** A **reducer** is a small function that says how to combine the **old value** of a key with a **new update**. By default LangGraph **overwrites** the old value. You attach a different reducer with `Annotated[type, reducer]`.

**Analogy.** A whiteboard. Without a reducer, every writer **erases** the board and writes their line. With `operator.add`, every writer **adds a line** at the bottom.

| Reducer | Written as | What it does | Typical use |
|---|---|---|---|
| (none) | `count: int` | new value replaces old | counters, flags, "current step" |
| `operator.add` | `Annotated[list, operator.add]` | `old + new` (lists are joined) | logs, collected results from parallel workers |
| `add_messages` | `Annotated[list, add_messages]` | appends messages, but **replaces** a message that has the same `id`, and deletes on `RemoveMessage` | chat history |
| your own | `Annotated[dict, my_merge]` | anything you want | merging dicts, keeping the max, de-duplicating |

`MessagesState` is a ready-made state with one key, `messages: Annotated[list[AnyMessage], add_messages]`.

**Why it matters: parallel nodes.** If two nodes run in the **same** round and both write the same key, LangGraph needs to know how to combine them. Without a reducer it refuses and raises `InvalidUpdateError`.

In [3]:
from langgraph.errors import InvalidUpdateError

def build_fanout(state_cls):
    """START fans out to two nodes that run in parallel and both write 'notes'."""
    b = StateGraph(state_cls)
    b.add_node("left",  lambda s: {"notes": ["from left"]})
    b.add_node("right", lambda s: {"notes": ["from right"]})
    b.add_edge(START, "left");  b.add_edge(START, "right")   # two edges out of START = parallel
    b.add_edge("left", END);    b.add_edge("right", END)
    return b.compile()

class NoReducer(TypedDict):
    notes: list                                  # default: overwrite

class WithReducer(TypedDict):
    notes: Annotated[list, operator.add]         # reducer: join lists

try:
    build_fanout(NoReducer).invoke({"notes": []})
except InvalidUpdateError as e:
    print("Without a reducer ->", type(e).__name__, ":", str(e).splitlines()[0][:90])

print("With operator.add ->", build_fanout(WithReducer).invoke({"notes": []}))

Without a reducer -> InvalidUpdateError : At key 'notes': Can receive only one value per step. Use an Annotated key to handle multip
With operator.add -> {'notes': ['from left', 'from right']}


In [4]:
# add_messages: append, replace-by-id, and delete
history = [HumanMessage("hi", id="1"), AIMessage("helo (typo)", id="2")]

appended = add_messages(history, [HumanMessage("how are you?", id="3")])
print("append       :", [m.content for m in appended])

fixed = add_messages(appended, [AIMessage("hello (fixed)", id="2")])   # same id -> replaced
print("replace id=2 :", [m.content for m in fixed])

trimmed = add_messages(fixed, [RemoveMessage(id="1")])                 # delete by id
print("remove id=1  :", [m.content for m in trimmed])

# A custom reducer: keep the highest score ever seen.
def keep_max(old: float, new: float) -> float:
    return max(old, new)
class ScoreState(TypedDict):
    best: Annotated[float, keep_max]
print("custom reducer:", keep_max(0.4, 0.9), "(old 0.4, new 0.9)")

append       : ['hi', 'helo (typo)', 'how are you?']
replace id=2 : ['hi', 'hello (fixed)', 'how are you?']
remove id=1  : ['hello (fixed)', 'how are you?']
custom reducer: 0.9 (old 0.4, new 0.9)


## 4. Conditional edges and routing

**What it is.** A **conditional edge** runs a small **routing function** after a node. The function looks at the state and returns the **name of the next node** (or `END`). An optional **path map** translates the function's return value into node names.

**Analogy.** A receptionist who reads your form and sends you to "billing", "tech support" or "general".

**How it works.** `add_conditional_edges(source, route_fn, path_map)`. In a real app `route_fn` often reads a label that an LLM wrote into the state. Here a keyword rule stands in for the LLM.

In [5]:
class Ticket(TypedDict):
    text: str
    category: str
    reply: str

def classify(state: Ticket) -> dict:
    # Stand-in for an LLM classifier: simple keyword rules.
    t = state["text"].lower()
    cat = "billing" if ("refund" in t or "invoice" in t) else "tech" if ("error" in t or "crash" in t) else "general"
    return {"category": cat}

def route(state: Ticket) -> str:
    return state["category"]                       # returns a KEY of the path map below

b = StateGraph(Ticket)
b.add_node("classify", classify)
b.add_node("billing", lambda s: {"reply": "Billing team: we will check your invoice."})
b.add_node("tech",    lambda s: {"reply": "Tech team: please send the error log."})
b.add_node("general", lambda s: {"reply": "Thanks, a person will reply soon."})
b.add_edge(START, "classify")
b.add_conditional_edges("classify", route,
                        {"billing": "billing", "tech": "tech", "general": "general"})   # path map
for n in ["billing", "tech", "general"]:
    b.add_edge(n, END)
router_graph = b.compile()

for text in ["I want a refund", "The app shows an error", "Do you have stickers?"]:
    out = router_graph.invoke({"text": text})
    print(f"{text!r:28} -> {out['category']:8} -> {out['reply']}")

'I want a refund'            -> billing  -> Billing team: we will check your invoice.
'The app shows an error'     -> tech     -> Tech team: please send the error log.
'Do you have stickers?'      -> general  -> Thanks, a person will reply soon.


## 5. Cycles with a stop condition

**What it is.** A **cycle** is an edge that goes back to an earlier node, so steps can repeat ("try again", "call another tool"). Every cycle needs a **stop condition**, or it would run forever.

**Analogy.** Editing an essay: write, check, rewrite... until it is good enough **or** you run out of time.

**How it works.** A conditional edge decides "loop again" or `END`. Always use **two** stop rules: a quality rule ("good enough") **and** a safety rule ("at most N tries"). LangGraph also has a global safety net, `recursion_limit` (section 19).

In [6]:
class Draft(TypedDict):
    text: str
    attempts: int
    score: int

def write(state: Draft) -> dict:
    n = state["attempts"] + 1
    # Stand-in for an LLM writer: each attempt adds one more sentence.
    return {"text": state["text"] + f" Sentence {n}.", "attempts": n}

def grade(state: Draft) -> dict:
    # Stand-in for a grader: score = number of sentences x 3 (out of 10)
    return {"score": min(10, 3 * state["text"].count("Sentence"))}

MAX_TRIES = 5
def good_enough(state: Draft) -> str:
    if state["score"] >= 8:          # quality stop rule
        return "done"
    if state["attempts"] >= MAX_TRIES:   # safety stop rule
        return "done"
    return "again"

b = StateGraph(Draft)
b.add_node("write", write); b.add_node("grade", grade)
b.add_edge(START, "write"); b.add_edge("write", "grade")
b.add_conditional_edges("grade", good_enough, {"again": "write", "done": END})   # the cycle
loop_graph = b.compile()
print(loop_graph.invoke({"text": "", "attempts": 0, "score": 0}))

{'text': ' Sentence 1. Sentence 2. Sentence 3.', 'attempts': 3, 'score': 9}


## 6. Running a graph: invoke vs stream (and stream modes)

**What it is.** `invoke()` waits and returns the **final** state. `stream()` gives you results **while the graph runs**, which is what you want for progress bars and chat UIs. The `stream_mode` decides what each streamed chunk contains:

| `stream_mode` | Each chunk is... | Use it for |
|---|---|---|
| `"values"` | the **full state** after each step | debugging, "show me everything" |
| `"updates"` | `{node_name: what_that_node_returned}` | progress ("now running: grade") |
| `"messages"` | `(token_chunk, metadata)` pairs from LLM calls inside nodes | typing-effect chat UIs (section 20) |
| `"custom"` | anything a node sends with `get_stream_writer()` | your own progress events |
| `["updates", "custom"]` | `(mode, chunk)` tuples | several at once |

In [7]:
start = {"text": "", "attempts": 0, "score": 0}
print("stream_mode='updates'  (only what each node changed):")
for chunk in loop_graph.stream(start, stream_mode="updates"):
    print("  ", chunk)

print("\nstream_mode='values'  (full state after each step, first 3 only):")
for i, chunk in enumerate(loop_graph.stream(start, stream_mode="values")):
    if i < 3: print("  ", chunk)

stream_mode='updates'  (only what each node changed):
   {'write': {'text': ' Sentence 1.', 'attempts': 1}}
   {'grade': {'score': 3}}
   {'write': {'text': ' Sentence 1. Sentence 2.', 'attempts': 2}}
   {'grade': {'score': 6}}
   {'write': {'text': ' Sentence 1. Sentence 2. Sentence 3.', 'attempts': 3}}
   {'grade': {'score': 9}}

stream_mode='values'  (full state after each step, first 3 only):
   {'text': '', 'attempts': 0, 'score': 0}
   {'text': ' Sentence 1.', 'attempts': 1, 'score': 0}
   {'text': ' Sentence 1.', 'attempts': 1, 'score': 3}


In [8]:
from langgraph.config import get_stream_writer

class Job(TypedDict):
    files: list
    done: int

def process_files(state: Job) -> dict:
    writer = get_stream_writer()                  # lets a node emit custom events
    for i, f in enumerate(state["files"], 1):
        writer({"progress": f"{i}/{len(state['files'])}", "file": f})
    return {"done": len(state["files"])}

b = StateGraph(Job); b.add_node("process", process_files); b.add_edge(START, "process")
job_graph = b.compile()
for mode, chunk in job_graph.stream({"files": ["a.csv", "b.csv", "c.csv"], "done": 0},
                                    stream_mode=["custom", "updates"]):
    print(f"  [{mode}] {chunk}")

  [custom] {'progress': '1/3', 'file': 'a.csv'}
  [custom] {'progress': '2/3', 'file': 'b.csv'}
  [custom] {'progress': '3/3', 'file': 'c.csv'}
  [updates] {'process': {'done': 3}}


## 7. Visualising the graph

**What it is.** Every compiled graph can describe itself. `graph.get_graph().draw_mermaid()` returns **Mermaid** text (a small diagram language that GitHub, Jupyter and many docs tools can render). Solid arrows are fixed edges; dotted arrows are conditional edges.

`draw_mermaid_png()` also exists, but by default it calls a web service to render the picture, so we skip it here (no internet). `draw_ascii()` needs the extra `grandalf` package.

In [9]:
mermaid_text = loop_graph.get_graph().draw_mermaid()
print(mermaid_text)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	write(write)
	grade(grade)
	__end__([<p>__end__</p>]):::last
	__start__ --> write;
	grade -. &nbsp;done&nbsp; .-> __end__;
	grade -. &nbsp;again&nbsp; .-> write;
	write --> grade;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



Pasted into a Markdown cell, that text renders like this (the repo's notebooks render Mermaid blocks):

```mermaid
graph TD;
    __start__([__start__]) --> write;
    write --> grade;
    grade -. again .-> write;
    grade -. done .-> __end__([__end__]);
```

## 8. A tool-calling agent, built by hand (ToolNode + tools_condition)

**What it is.** The classic agent loop as a graph:

```mermaid
flowchart LR
    S([START]) --> A[agent: LLM decides]
    A -- "AIMessage has tool_calls" --> T[tools: run them]
    T --> A
    A -- "no tool_calls" --> E([END])
```

- A **tool** is a Python function the LLM may ask to run. `@tool` turns a function into a tool; its docstring and type hints become the description the LLM reads.
- A **tool call** is the LLM's request: an `AIMessage` with `tool_calls=[{"name", "args", "id"}]`.
- **`ToolNode`** is a prebuilt node that runs every tool call in the last `AIMessage` and returns one `ToolMessage` per call.
- **`tools_condition`** is a prebuilt routing function: it returns `"tools"` if the last message has tool calls, else `END`.

**Analogy.** A manager (LLM) writes job slips ("look up the weather in Paris"); an assistant (ToolNode) does the jobs and staples the answers to the slips; the manager reads them and either writes more slips or gives the final answer.

The LLM here is our **fake** `ScriptedChatModel`: we script exactly what a real model would typically reply.

In [10]:
from langgraph.prebuilt import ToolNode, tools_condition

@tool
def get_weather(city: str) -> str:
    """Return today's weather for a city."""
    fake_db = {"paris": "18C, cloudy", "tokyo": "24C, sunny"}     # toy data, no internet
    return fake_db.get(city.lower(), "unknown")

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

tools = [get_weather, multiply]

def scripted_replies():
    # What a real LLM might answer for: "Weather in Paris and Tokyo, and what is 6*7?"
    return iter([
        AIMessage("", tool_calls=[
            {"name": "get_weather", "args": {"city": "Paris"}, "id": "call_1"},
            {"name": "get_weather", "args": {"city": "Tokyo"}, "id": "call_2"},
            {"name": "multiply", "args": {"a": 6, "b": 7}, "id": "call_3"}]),
        AIMessage("Paris is 18C and cloudy, Tokyo is 24C and sunny, and 6 x 7 = 42."),
    ])

llm = ScriptedChatModel(messages=scripted_replies()).bind_tools(tools)

def agent(state: MessagesState) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

b = StateGraph(MessagesState)
b.add_node("agent", agent)
b.add_node("tools", ToolNode(tools))
b.add_edge(START, "agent")
b.add_conditional_edges("agent", tools_condition)   # -> "tools" or END
b.add_edge("tools", "agent")                        # the loop
tool_agent = b.compile()

out = tool_agent.invoke({"messages": [HumanMessage("Weather in Paris and Tokyo, and what is 6*7?")]})
show(out["messages"])

  HumanMessage  'Weather in Paris and Tokyo, and what is 6*7?'
  AIMessage     ''  tool_calls=[('get_weather', {'city': 'Paris'}), ('get_weather', {'city': 'Tokyo'}), ('multiply', {'a': 6, 'b': 7})]
  ToolMessage   '18C, cloudy'
  ToolMessage   '24C, sunny'
  ToolMessage   '42.0'
  AIMessage     'Paris is 18C and cloudy, Tokyo is 24C and sunny, and 6 x 7 = 42.'


**What just happened?**
1. `agent` returned an `AIMessage` with **three** tool calls.
2. `tools_condition` saw tool calls, so it routed to `tools`.
3. `ToolNode` ran all three and returned three `ToolMessage`s (each linked to its call by `tool_call_id`).
4. Back in `agent`, the (fake) LLM read the results and answered with no tool calls, so `tools_condition` routed to `END`.

**Tool errors.** `ToolNode(tools, handle_tool_errors=...)` controls what happens when a tool fails. In recent versions the default only turns **invalid arguments** into an error `ToolMessage` (so the LLM can fix its call) and re-raises other exceptions; pass `handle_tool_errors=True` (or your own function) to send every error back to the LLM instead of crashing the graph. Defaults have changed between versions, so set it explicitly.

## 9. The prebuilt agent: `create_react_agent`

**What it is.** A one-line factory that builds exactly the graph from section 8 (agent node + ToolNode + loop). **ReAct** ("reason + act", Yao et al., 2022) is the name of this think-act-observe loop.

**Version note (as of late 2026).** In LangGraph 1.x, `langgraph.prebuilt.create_react_agent` still works but is **deprecated**: the recommended successor is `create_agent` in the `langchain` package (`from langchain.agents import create_agent`), which adds "middleware" hooks. Both build the same kind of loop. We use `create_react_agent` because it ships with LangGraph, and we hide the deprecation warning.

| Option | What it does |
|---|---|
| `prompt=` | a system prompt (string or `SystemMessage`) |
| `checkpointer=` | memory across turns (section 10) |
| `response_format=` | ask for a final structured (Pydantic) answer |
| `pre_model_hook=` / `post_model_hook=` | run code before/after each LLM call (e.g. trim history, guardrails) |
| `interrupt_before=["tools"]` | pause before running tools so a human can approve |

In [11]:
from langgraph.prebuilt import create_react_agent

with warnings.catch_warnings():
    warnings.simplefilter("ignore")          # hide the "moved to langchain.agents" notice
    react_agent = create_react_agent(
        ScriptedChatModel(messages=scripted_replies()),
        tools,
        prompt="You are a helpful assistant. Use tools when needed.",
    )

out = react_agent.invoke({"messages": [HumanMessage("Weather in Paris and Tokyo, and what is 6*7?")]})
show(out["messages"])
print("\nNodes in the prebuilt graph:", list(react_agent.get_graph().nodes))

  HumanMessage  'Weather in Paris and Tokyo, and what is 6*7?'
  AIMessage     ''  tool_calls=[('get_weather', {'city': 'Paris'}), ('get_weather', {'city': 'Tokyo'}), ('multiply', {'a': 6, 'b': 7})]
  ToolMessage   '18C, cloudy'
  ToolMessage   '24C, sunny'
  ToolMessage   '42.0'
  AIMessage     'Paris is 18C and cloudy, Tokyo is 24C and sunny, and 6 x 7 = 42.'

Nodes in the prebuilt graph: ['__start__', 'agent', 'tools', '__end__']


## 10. Checkpointers and `thread_id`: memory across turns

**What it is.**
- A **checkpointer** saves a **checkpoint** (a snapshot of the whole state) after every step.
- A **thread** is one conversation. You pick its id with `config={"configurable": {"thread_id": "..."}}`. Calling the graph again with the same `thread_id` continues from the last checkpoint; a new id starts fresh.

**Analogy.** A video game "save slot". Each player (thread) has their own slot; loading a slot puts you exactly where you left off.

**Which checkpointer?**

| Checkpointer | Where it saves | Use for |
|---|---|---|
| `InMemorySaver` (old name `MemorySaver`, still works) | Python memory, lost on restart | notebooks, tests |
| `SqliteSaver` (package `langgraph-checkpoint-sqlite`) | a local SQLite file | small apps, local tools |
| `PostgresSaver` (package `langgraph-checkpoint-postgres`) | Postgres | production, many users |

Checkpoints are also what makes **human-in-the-loop**, **time travel** and **crash recovery** ("durable execution") possible.

In [12]:
def memory_bot(state: MessagesState) -> dict:
    # Stand-in for an LLM: it can only answer from the conversation history it is given.
    last = state["messages"][-1].content.lower()
    name = None
    for m in state["messages"]:
        if isinstance(m, HumanMessage) and "my name is" in m.content.lower():
            name = m.content.lower().split("my name is")[1].strip(" .!").title()
    if "my name is" in last:
        reply = f"Nice to meet you, {name}."
    elif "what is my name" in last:
        reply = f"Your name is {name}." if name else "I don't know your name yet."
    else:
        reply = "OK."
    return {"messages": [AIMessage(reply)]}

b = StateGraph(MessagesState); b.add_node("bot", memory_bot); b.add_edge(START, "bot")
saver = InMemorySaver()
chat = b.compile(checkpointer=saver)

alice = {"configurable": {"thread_id": "alice"}}
bob   = {"configurable": {"thread_id": "bob"}}
print("alice:", chat.invoke({"messages": [HumanMessage("Hi, my name is Alice")]}, alice)["messages"][-1].content)
print("alice:", chat.invoke({"messages": [HumanMessage("What is my name?")]}, alice)["messages"][-1].content)
print("bob  :", chat.invoke({"messages": [HumanMessage("What is my name?")]}, bob)["messages"][-1].content)

snap = chat.get_state(alice)          # the latest checkpoint of a thread
print("\nalice's thread has", len(snap.values["messages"]), "messages; next node:", snap.next)

alice: Nice to meet you, Alice.
alice: Your name is Alice.
bob  : I don't know your name yet.

alice's thread has 4 messages; next node: ()


## 11. Time travel: history, replay and fork

**What it is.** Because every step is checkpointed, you can:
- **look back**: `graph.get_state_history(config)` lists every checkpoint of a thread (newest first);
- **replay**: `graph.invoke(None, past_checkpoint.config)` re-runs from that point (steps **before** it are not re-run);
- **fork**: `graph.update_state(past_config, {...})` writes an edited copy of a past state as a **new** checkpoint, then `invoke(None, new_config)` continues from it. The original history is kept.

**Analogy.** "Undo history" in a document editor, plus "save as a new version from here".

**When to use it.** Debugging ("what did the state look like before the bad tool call?"), trying "what if the LLM had said X instead?", and recovering from a bad step without starting over.

In [13]:
class Calc(TypedDict):
    n: int
    log: Annotated[list, operator.add]

b = StateGraph(Calc)
b.add_node("add_one", lambda s: {"n": s["n"] + 1, "log": [f"add_one -> {s['n'] + 1}"]})
b.add_node("double",  lambda s: {"n": s["n"] * 2, "log": [f"double -> {s['n'] * 2}"]})
b.add_edge(START, "add_one"); b.add_edge("add_one", "double"); b.add_edge("double", END)
calc = b.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "tt-1"}}
print("first run:", calc.invoke({"n": 1, "log": []}, cfg))

print("\nHistory (newest first):")
history = list(calc.get_state_history(cfg))
for h in history:
    print(f"  id=...{h.config['configurable']['checkpoint_id'][-6:]}  next={h.next!s:14} values={h.values}")

first run: {'n': 4, 'log': ['add_one -> 2', 'double -> 4']}

History (newest first):
  id=...f06b90  next=()             values={'n': 4, 'log': ['add_one -> 2', 'double -> 4']}
  id=...9897b9  next=('double',)    values={'n': 2, 'log': ['add_one -> 2']}
  id=...01cf99  next=('add_one',)   values={'n': 1, 'log': []}
  id=...435a10  next=('__start__',) values={'log': []}


In [14]:
# Pick the checkpoint taken just BEFORE "double" ran.
before_double = next(h for h in history if h.next == ("double",))

# Replay: continue from that checkpoint (add_one is NOT re-run).
print("replay:", calc.invoke(None, before_double.config))

# Fork: edit the past state (n=2 -> n=100) and continue on a new branch.
fork_cfg = calc.update_state(before_double.config, {"n": 100})
print("fork  :", calc.invoke(None, fork_cfg))
print("checkpoints in this thread now:", len(list(calc.get_state_history(cfg))))

replay: {'n': 4, 'log': ['add_one -> 2', 'double -> 4']}
fork  : {'n': 200, 'log': ['add_one -> 2', 'double -> 200']}
checkpoints in this thread now: 8


## 12. Human-in-the-loop: `interrupt()` and `Command(resume=...)`

**What it is.** **Human-in-the-loop (HITL)** means the graph **pauses** and waits for a person to approve, edit or answer before it continues.
- Inside a node, `answer = interrupt(payload)` pauses the run. `payload` is what you show the human (a question, a draft, a tool call).
- The run returns early with a special `__interrupt__` key. The state is saved by the checkpointer (so a checkpointer is **required**).
- Later (seconds or days), you continue with `graph.invoke(Command(resume=value), config)`. Now `interrupt(...)` **returns** `value`.

**Analogy.** A form that needs a manager's signature. The clerk fills it in, puts it in the "waiting for signature" tray (checkpoint), and goes home. When the manager signs, the clerk picks it up exactly where they left off.

**Important rule:** on resume, the paused node **runs again from its first line**. Any code before `interrupt()` runs twice, so keep side effects (sending emails, charging cards) **after** the interrupt, or in a separate node. The counter below proves this.

```mermaid
flowchart LR
    S([START]) --> P[propose_refund] --> A{{approve: interrupt}}
    A -- "resume: yes" --> R[issue_refund] --> E([END])
    A -- "resume: no" --> X[cancel] --> E
```

In [15]:
class Refund(TypedDict):
    order_id: str
    amount: float
    decision: str
    result: str

calls_to_approve = {"count": 0}      # to prove the node re-runs on resume

def propose(state: Refund) -> dict:
    return {"amount": 250.0}         # stand-in: an LLM decided to refund 250

def approve(state: Refund) -> Command[Literal["issue_refund", "cancel"]]:
    calls_to_approve["count"] += 1   # code BEFORE interrupt() runs twice!
    if state["amount"] <= 100:       # small refunds are auto-approved
        return Command(goto="issue_refund", update={"decision": "auto"})
    answer = interrupt({"question": f"Approve refund of ${state['amount']} for {state['order_id']}?"})
    return Command(goto="issue_refund" if answer == "yes" else "cancel", update={"decision": answer})

b = StateGraph(Refund)
b.add_node("propose", propose)
b.add_node("approve", approve)
b.add_node("issue_refund", lambda s: {"result": f"refunded ${s['amount']}"})
b.add_node("cancel",       lambda s: {"result": "refund cancelled"})
b.add_edge(START, "propose"); b.add_edge("propose", "approve")
b.add_edge("issue_refund", END); b.add_edge("cancel", END)
refund_graph = b.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "refund-42"}}
paused = refund_graph.invoke({"order_id": "A-42"}, cfg)
print("paused with:", paused["__interrupt__"][0].value)
print("waiting at node:", refund_graph.get_state(cfg).next)

paused with: {'question': 'Approve refund of $250.0 for A-42?'}
waiting at node: ('approve',)


In [16]:
# ... later, a human clicks "yes" in a UI:
final = refund_graph.invoke(Command(resume="yes"), cfg)
print("after resume:", final["result"], "| decision:", final["decision"])
print("approve() ran", calls_to_approve["count"], "times (once before the pause, once on resume)")

# Another thread where the human says no:
cfg2 = {"configurable": {"thread_id": "refund-43"}}
refund_graph.invoke({"order_id": "A-43"}, cfg2)
print("thread 2:", refund_graph.invoke(Command(resume="no"), cfg2)["result"])

after resume: refunded $250.0 | decision: yes
approve() ran 2 times (once before the pause, once on resume)


thread 2: refund cancelled


**Common HITL designs**

| Pattern | How | Example |
|---|---|---|
| Approve / reject | `interrupt()` returns "yes"/"no", route with `Command(goto=...)` | refunds, deleting files, sending emails |
| Edit state | resume with an edited value and write it into the state | fix a draft before it is published |
| Review a tool call | interrupt in a node before `ToolNode`, or compile with `interrupt_before=["tools"]` | "the agent wants to run `rm -rf`, allow?" |
| Ask the user | `interrupt("Which city?")` used as a question | missing information |

`interrupt_before` / `interrupt_after` are **static breakpoints** (pause before/after a named node every time). They are handy for debugging; `interrupt()` is the more flexible, recommended way for real approvals.

## 13. `Command`: update the state and choose the next node in one step

**What it is.** A node can return `Command(update={...}, goto="next_node")` instead of a plain dict. It does two jobs at once: **update the state** and **jump** to a node. This replaces "node + separate conditional edge" when the node itself knows where to go.

**Analogy.** A runner who hands over the baton **and** points at which teammate runs next.

**How it works.**
- Type-hint the return as `Command[Literal["a", "b"]]` so LangGraph knows the possible targets (used for drawing and checks).
- `goto` can also be a list (run several next) or a `Send` (section 14).
- `Command(graph=Command.PARENT, goto=...)` jumps to a node in the **parent** graph from inside a subgraph. This is how agent "handoffs" work in multi-agent graphs.
- `Command(resume=...)` is the same class used to resume an interrupt (section 12).

| Use a conditional edge when... | Use `Command` when... |
|---|---|
| routing is a pure function of the state | the node computes something and decides the route at the same time |
| you want the routing logic visible in one place | agents hand off to each other (multi-agent) |

In [17]:
class Order(TypedDict):
    items: int
    status: Annotated[list, operator.add]

def check_stock(state: Order) -> Command[Literal["ship", "backorder"]]:
    in_stock = state["items"] <= 3                       # toy rule
    return Command(update={"status": [f"checked {state['items']} items"]},
                   goto="ship" if in_stock else "backorder")

b = StateGraph(Order)
b.add_node("check_stock", check_stock)
b.add_node("ship",      lambda s: {"status": ["shipped"]})
b.add_node("backorder", lambda s: {"status": ["back-ordered"]})
b.add_edge(START, "check_stock"); b.add_edge("ship", END); b.add_edge("backorder", END)
order_graph = b.compile()          # note: no add_conditional_edges needed
print(order_graph.invoke({"items": 2, "status": []}))
print(order_graph.invoke({"items": 9, "status": []}))

{'items': 2, 'status': ['checked 2 items', 'shipped']}
{'items': 9, 'status': ['checked 9 items', 'back-ordered']}


## 14. The Send API: parallel fan-out (map-reduce)

**What it is.** Sometimes you only know **at run time** how many parallel jobs you need (one per document, one per sub-question). A routing function can return a list of `Send("node_name", payload)` objects. Each `Send` starts a **separate copy** of that node with its own small input. Their outputs are merged with a reducer.

- **Map** = run the same step on many items in parallel.
- **Reduce** = combine all the results into one.

**Analogy.** A teacher hands one chapter to each student (map), then collects the summaries and writes the class summary (reduce).

```mermaid
flowchart LR
    S([START]) --> P[plan: split into docs]
    P -- "Send(doc 1)" --> W1[summarise]
    P -- "Send(doc 2)" --> W2[summarise]
    P -- "Send(doc 3)" --> W3[summarise]
    W1 --> R[reduce: combine]
    W2 --> R
    W3 --> R
    R --> E([END])
```

In [18]:
class MapReduce(TypedDict):
    docs: list
    summaries: Annotated[list, operator.add]     # reducer collects parallel results
    final: str

class OneDoc(TypedDict):                         # the small private input of each worker
    doc: str

def fan_out(state: MapReduce):
    # one Send per document; the number is decided at run time
    return [Send("summarise", {"doc": d}) for d in state["docs"]]

def summarise(state: OneDoc) -> dict:
    # Stand-in for an LLM summary: the first 4 words.
    return {"summaries": [" ".join(state["doc"].split()[:4]) + "..."]}

def reduce_step(state: MapReduce) -> dict:
    return {"final": " | ".join(state["summaries"])}

b = StateGraph(MapReduce)
b.add_node("summarise", summarise)
b.add_node("reduce", reduce_step)
b.add_conditional_edges(START, fan_out, ["summarise"])   # list = possible targets
b.add_edge("summarise", "reduce")
b.add_edge("reduce", END)
mr_graph = b.compile()

docs = ["LangGraph models agents as graphs of nodes and edges.",
        "Checkpointers save the state after every step.",
        "The Send API runs the same node on many inputs in parallel."]
out = mr_graph.invoke({"docs": docs, "summaries": []})
print(out["final"])

LangGraph models agents as... | Checkpointers save the state... | The Send API runs...


## 15. Subgraphs: a graph inside a graph

**What it is.** A **subgraph** is a compiled graph used as a node of another graph. It lets you build big systems from small, separately tested parts (for example one subgraph per agent team).

**Analogy.** A company department. From outside it looks like one box ("Legal, please review this"); inside it has its own steps and people.

There are two ways to plug one in:

| Way | When | How |
|---|---|---|
| **Add the compiled subgraph directly as a node** | parent and child **share** state keys | `parent.add_node("team", child_graph)` |
| **Call it from inside a normal node** | the child has a **different** state shape | `def node(s): out = child.invoke({...convert...}); return {...convert back...}` |

With a checkpointer on the parent, subgraphs are checkpointed too, and `stream(..., subgraphs=True)` shows their inner steps.

**Gotcha.** When you add a subgraph directly (way 1), it hands back the **whole** value of each shared key. If that key has an `operator.add` reducer and the child received a non-empty list, the parent appends the old items **again** (duplicates). Fix: start the child with an empty list (way 2), or give the child its own output key.

In [19]:
# Child graph: a tiny "research team" with its own private key 'raw'
class TeamState(TypedDict):
    topic: str
    notes: Annotated[list, operator.add]     # shared with the parent
    raw: str                                 # private to the child

child = StateGraph(TeamState)
child.add_node("search", lambda s: {"raw": f"3 facts about {s['topic']}"})
child.add_node("clean",  lambda s: {"notes": [f"research: {s['raw']}"]})
child.add_edge(START, "search"); child.add_edge("search", "clean")
research_team = child.compile()

# Parent graph: uses the child as ONE node (shared keys: topic, notes)
class ProjectState(TypedDict):
    topic: str
    notes: Annotated[list, operator.add]

parent = StateGraph(ProjectState)
parent.add_node("research_team", research_team)                       # way 1: direct
parent.add_node("write", lambda s: {"notes": [f"draft based on {len(s['notes'])} note(s)"]})
parent.add_edge(START, "research_team"); parent.add_edge("research_team", "write")
project = parent.compile()

for ns, update in project.stream({"topic": "graphs", "notes": []}, stream_mode="updates", subgraphs=True):
    where = "inside " + ns[0].split(":")[0] if ns else "parent"
    print(f"  [{where:22}] {update}")

  [inside research_team  ] {'search': {'raw': '3 facts about graphs'}}
  [inside research_team  ] {'clean': {'notes': ['research: 3 facts about graphs']}}
  [parent                ] {'research_team': {'topic': 'graphs', 'notes': ['research: 3 facts about graphs']}}
  [parent                ] {'write': {'notes': ['draft based on 1 note(s)']}}


In [20]:
# Way 2: the child has a DIFFERENT state, so we convert inside a normal node.
class Translator(TypedDict):
    english: str
    french: str

t = StateGraph(Translator)
t.add_node("translate", lambda s: {"french": {"hello": "bonjour", "thanks": "merci"}.get(s["english"], "?")})
t.add_edge(START, "translate")
translator = t.compile()

def call_translator(state: ProjectState) -> dict:
    out = translator.invoke({"english": "thanks"})        # convert parent -> child
    return {"notes": [f"translated: {out['french']}"]}    # convert child -> parent

p2 = StateGraph(ProjectState); p2.add_node("translate_step", call_translator); p2.add_edge(START, "translate_step")
print(p2.compile().invoke({"topic": "x", "notes": []}))

{'topic': 'x', 'notes': ['translated: merci']}


## 16. Long-term memory with the Store

**What it is.** A checkpointer remembers **one thread**. A **Store** remembers things **across threads** (across conversations), for example "this user prefers short answers". Items are saved under a **namespace** (a tuple like `("user_42", "preferences")`, similar to a folder path) and a **key**.

| | Checkpointer | Store |
|---|---|---|
| Scope | one thread (conversation) | shared across threads |
| Holds | the full graph state at each step | small JSON documents you choose to save |
| Typical content | chat history, current step | user profile, learned facts, past summaries |
| Classes | `InMemorySaver`, `SqliteSaver`, `PostgresSaver` | `InMemoryStore`, `PostgresStore` |

**Analogy.** The checkpointer is your notebook for *today's* meeting. The store is the company wiki everyone can look things up in later.

**How it works.** `store.put(namespace, key, value)`, `store.get(namespace, key)`, `store.search(namespace_prefix, query=..., filter=..., limit=...)`. If you give the store an `index` with an embedding function, `search(query=...)` ranks items by **meaning** (vector similarity). An **embedding** is a list of numbers that represents the meaning of a text. Here a toy word-presence embedding stands in for a real embedding model. See [notebook 11](11_context_engineering_memory_and_agent_protocols.ipynb) for memory design (what to save, when to forget).

In [21]:
VOCAB = ["tea", "coffee", "python", "java", "short", "long", "dog", "cat"]
def toy_embed(texts: list[str]) -> list[list[float]]:
    """Stand-in for an embedding model: 1.0 if a vocabulary word appears, else 0.0."""
    return [[1.0 if w in t.lower() else 0.0 for w in VOCAB] for t in texts]

store = InMemoryStore(index={"dims": len(VOCAB), "embed": toy_embed})
ns = ("user_42", "facts")
store.put(ns, "drink", {"text": "User drinks tea every morning"})
store.put(ns, "lang",  {"text": "User writes python at work"})
store.put(ns, "style", {"text": "User prefers short answers"})

print("get   :", store.get(ns, "lang").value)
for item in store.search(ns, query="which programming language, python or java?", limit=2):
    print(f"search: score={item.score:.2f}  {item.value['text']}")

get   : {'text': 'User writes python at work'}


search: score=0.71  User writes python at work
search: score=0.00  User drinks tea every morning


In [22]:
# Using the store INSIDE a graph: nodes receive it through the `runtime` argument.
from dataclasses import dataclass
from langgraph.runtime import Runtime

@dataclass
class UserContext:              # "context" = per-run settings that are not part of the state
    user_id: str

def remember_or_recall(state: MessagesState, runtime: Runtime[UserContext]) -> dict:
    text = state["messages"][-1].content
    ns = (runtime.context.user_id, "facts")
    if text.lower().startswith("remember:"):
        fact = text.split(":", 1)[1].strip()
        runtime.store.put(ns, str(uuid.uuid4())[:8], {"text": fact})
        return {"messages": [AIMessage(f"Saved: {fact}")]}
    hits = runtime.store.search(ns, query=text, limit=1)
    return {"messages": [AIMessage(f"From memory: {hits[0].value['text']}" if hits else "Nothing saved yet.")]}

b = StateGraph(MessagesState, context_schema=UserContext)
b.add_node("memory", remember_or_recall); b.add_edge(START, "memory")
mem_graph = b.compile(checkpointer=InMemorySaver(), store=InMemoryStore(index={"dims": len(VOCAB), "embed": toy_embed}))

ask = lambda text, thread: mem_graph.invoke({"messages": [HumanMessage(text)]},
                                            {"configurable": {"thread_id": thread}},
                                            context=UserContext(user_id="u1"))["messages"][-1].content
print("thread A:", ask("remember: I like my coffee black", "A"))
print("thread B:", ask("how do I take my coffee?", "B"))     # a DIFFERENT thread still finds it

thread A: Saved: I like my coffee black
thread B: From memory: I like my coffee black


## 17. Multi-agent patterns, implemented for real

**What it is.** A **multi-agent system** splits work between several specialised agents (each with its own prompt, tools and maybe its own model). In LangGraph each agent is simply a **node** (or a subgraph), and the pattern is just **how the edges are wired**.

| Pattern | Who decides what runs next | Picture | Good for |
|---|---|---|---|
| **Supervisor** | one central "boss" agent | hub and spokes | clear tasks with specialist workers |
| **Handoffs / swarm** | the active agent hands control to a peer | relay | customer support flows (triage -> billing -> tech) |
| **Hierarchical** | supervisors of supervisors | org chart | big systems with teams |
| **Plan-and-execute** | a planner writes all steps first | to-do list | long tasks with clear steps |
| **Reflection** | a critic reviews the generator's output | writer + editor | quality-sensitive outputs (code, essays) |

All "LLMs" below are **scripted Python functions** so the control flow is easy to see. In a real system, each would be an LLM call that returns the same kind of decision (often as structured output). LangChain also publishes ready-made helpers (`langgraph-supervisor`, `langgraph-swarm` packages); they build graphs very similar to the ones we write by hand here.

### 17.1 Supervisor

The supervisor looks at the state and picks the next worker (or `FINISH`). Workers always report back to the supervisor.

```mermaid
flowchart LR
    S([START]) --> SUP{supervisor}
    SUP --> R[researcher] --> SUP
    SUP --> C[coder] --> SUP
    SUP --> W[writer] --> SUP
    SUP -- FINISH --> E([END])
```

In [23]:
class SupervisorState(TypedDict):
    task: str
    work: Annotated[list, operator.add]     # what each worker produced
    next: str

def supervisor(state: SupervisorState) -> Command[Literal["researcher", "coder", "writer", "__end__"]]:
    # Stand-in for an LLM supervisor: pick the first missing step.
    done = {w.split(":")[0] for w in state["work"]}
    for worker in ["researcher", "coder", "writer"]:
        if worker not in done:
            print(f"  supervisor -> {worker}")
            return Command(goto=worker, update={"next": worker})
    print("  supervisor -> FINISH")
    return Command(goto=END, update={"next": "FINISH"})

def researcher(state): return {"work": ["researcher: CSV has columns date, sales"]}
def coder(state):      return {"work": ["coder: df.groupby('month').sales.sum()"]}
def writer(state):     return {"work": ["writer: Sales rose 12% in March (toy numbers)."]}

b = StateGraph(SupervisorState)
b.add_node("supervisor", supervisor)
for name, fn in [("researcher", researcher), ("coder", coder), ("writer", writer)]:
    b.add_node(name, fn)
    b.add_edge(name, "supervisor")           # workers always report back
b.add_edge(START, "supervisor")
supervisor_graph = b.compile()

out = supervisor_graph.invoke({"task": "Analyse monthly sales", "work": []})
for w in out["work"]: print("   ", w)

  supervisor -> researcher
  supervisor -> coder
  supervisor -> writer
  supervisor -> FINISH
    researcher: CSV has columns date, sales
    coder: df.groupby('month').sales.sum()
    writer: Sales rose 12% in March (toy numbers).


### 17.2 Handoffs (swarm-style)

There is **no boss**. The agent that is currently talking to the user can **hand off** to a peer. We remember the `active_agent` in the state, and a checkpointer keeps it between turns, so the next user message goes straight to whoever is active.

**Analogy.** A phone call being transferred: "Let me put you through to billing." The next time you speak, you are talking to billing, not the receptionist.

In [24]:
class SwarmState(MessagesState):
    active_agent: str

def entry(state: SwarmState) -> Command[Literal["triage", "billing", "tech"]]:
    # Route each new user turn to whoever is currently active (default: triage)
    return Command(goto=state.get("active_agent") or "triage")

def triage(state: SwarmState) -> Command[Literal["billing", "tech", "__end__"]]:
    text = state["messages"][-1].content.lower()
    if "invoice" in text or "charge" in text:
        return Command(goto="billing", update={"active_agent": "billing",
                       "messages": [AIMessage("[triage] Transferring you to billing.")]})
    if "crash" in text:
        return Command(goto="tech", update={"active_agent": "tech",
                       "messages": [AIMessage("[triage] Transferring you to tech support.")]})
    return Command(goto=END, update={"messages": [AIMessage("[triage] How can I help?")]})

def billing(state: SwarmState) -> Command[Literal["tech", "__end__"]]:
    text = state["messages"][-1].content.lower()
    if "crash" in text:                                   # billing can hand off too
        return Command(goto="tech", update={"active_agent": "tech",
                       "messages": [AIMessage("[billing] That's a tech issue, transferring.")]})
    if "fixed" in text or "thanks" in text:
        return Command(goto=END, update={"messages": [AIMessage("[billing] Yes, the refund is complete.")]})
    return Command(goto=END, update={"messages": [AIMessage("[billing] I refunded the double charge.")]})

def tech(state: SwarmState) -> Command[Literal["__end__"]]:
    return Command(goto=END, update={"messages": [AIMessage("[tech] Please update to the latest version.")]})

b = StateGraph(SwarmState)
for name, fn in [("entry", entry), ("triage", triage), ("billing", billing), ("tech", tech)]:
    b.add_node(name, fn)
b.add_edge(START, "entry")
swarm = b.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "customer-7"}}
for user_text in ["I was charged twice on my invoice", "thanks, is it fixed?", "also the app crashes on start"]:
    out = swarm.invoke({"messages": [HumanMessage(user_text)]}, cfg)
    print(f"user: {user_text!r:40} -> {out['messages'][-1].content}   (active: {out['active_agent']})")

user: 'I was charged twice on my invoice'      -> [billing] I refunded the double charge.   (active: billing)
user: 'thanks, is it fixed?'                   -> [billing] Yes, the refund is complete.   (active: billing)
user: 'also the app crashes on start'          -> [tech] Please update to the latest version.   (active: tech)


### 17.3 Hierarchical teams (short)

A **top supervisor** delegates to **team subgraphs**, and each team has its own supervisor and workers. Here each team is a compiled subgraph (section 15), called from a small wrapper node so the shared `work` list is not duplicated.

```mermaid
flowchart TB
    TOP{top supervisor} --> RT[research team subgraph]
    TOP --> WT[writing team subgraph]
    RT --> TOP
    WT --> TOP
```

In [25]:
class OrgState(TypedDict):
    work: Annotated[list, operator.add]

def make_team(team: str, workers: list[str]):
    """Build a small team subgraph (workers run one after another) and wrap it
    in a node that starts the team with an EMPTY work list (see the gotcha in section 15)."""
    t = StateGraph(OrgState)
    prev = START
    for w in workers:
        t.add_node(w, lambda s, w=w: {"work": [f"{team}/{w}: done"]})
        t.add_edge(prev, w); prev = w
    team_graph = t.compile()
    def run_team(state: OrgState) -> dict:
        return {"work": team_graph.invoke({"work": []})["work"]}   # only the NEW items
    return run_team

def top_supervisor(state: OrgState) -> Command[Literal["research_team", "writing_team", "__end__"]]:
    done = " ".join(state["work"])
    if "research_team" not in done: return Command(goto="research_team")
    if "writing_team" not in done:  return Command(goto="writing_team")
    return Command(goto=END)

b = StateGraph(OrgState)
b.add_node("top", top_supervisor)
b.add_node("research_team", make_team("research_team", ["searcher", "fact_checker"]))
b.add_node("writing_team",  make_team("writing_team",  ["drafter", "editor"]))
b.add_edge(START, "top"); b.add_edge("research_team", "top"); b.add_edge("writing_team", "top")
print(b.compile().invoke({"work": []})["work"])

['research_team/searcher: done', 'research_team/fact_checker: done', 'writing_team/drafter: done', 'writing_team/editor: done']


### 17.4 Plan-and-execute

A **planner** writes the whole list of steps first. An **executor** does one step at a time. A **replanner** looks at the results and either finishes or changes the remaining plan. This saves cost (the big planning model is called rarely; a cheaper model can execute) and makes progress easy to show to the user.

```mermaid
flowchart LR
    S([START]) --> P[planner] --> X[executor]
    X --> RP{replanner}
    RP -- steps left --> X
    RP -- done --> E([END])
```

In [26]:
class PlanState(TypedDict):
    goal: str
    plan: list                                   # remaining steps (overwritten each time)
    past_steps: Annotated[list, operator.add]    # (step, result) pairs
    answer: str

def planner(state: PlanState) -> dict:
    # Stand-in for an LLM planner
    return {"plan": ["find the population of France", "find the population of Spain", "add them"]}

FAKE_FACTS = {"france": 68, "spain": 48}         # millions, toy values for the demo
def executor(state: PlanState) -> dict:
    step = state["plan"][0]
    if "add" in step:
        result = sum(int(r.split()[0]) for _, r in state["past_steps"])
        result = f"{result} million"
    else:
        country = step.split()[-1].lower()
        result = f"{FAKE_FACTS[country]} million"
    print(f"  executing: {step!r:38} -> {result}")
    return {"past_steps": [(step, result)], "plan": state["plan"][1:]}

def replanner(state: PlanState) -> Command[Literal["executor", "__end__"]]:
    if state["plan"]:                              # steps left: keep going
        return Command(goto="executor")
    return Command(goto=END, update={"answer": state["past_steps"][-1][1]})

b = StateGraph(PlanState)
b.add_node("planner", planner); b.add_node("executor", executor); b.add_node("replanner", replanner)
b.add_edge(START, "planner"); b.add_edge("planner", "executor"); b.add_edge("executor", "replanner")
out = b.compile().invoke({"goal": "Combined population of France and Spain?", "past_steps": []})
print("answer:", out["answer"], "(toy numbers)")

  executing: 'find the population of France'        -> 68 million
  executing: 'find the population of Spain'         -> 48 million
  executing: 'add them'                             -> 116 million
answer: 116 million (toy numbers)


### 17.5 Reflection loop (generator + critic)

A **generator** writes, a **critic** reviews and gives feedback, and the generator revises. Stop when the critic approves **or** after N rounds. This is a cycle (section 5) with two roles. Related research: Reflexion (Shinn et al., 2023) and Self-Refine (Madaan et al., 2023).

In [27]:
class ReflectState(TypedDict):
    draft: str
    feedback: Annotated[list, operator.add]
    rounds: int

def generator(state: ReflectState) -> dict:
    # Stand-in for an LLM that applies the latest feedback
    draft = state["draft"] or "def add(a, b): return a+b"
    if state["feedback"]:
        fb = state["feedback"][-1]
        if "docstring" in fb: draft = draft.replace(": return", ':\n    """Add two numbers."""\n    return', 1)
        if "type hints" in fb: draft = draft.replace("add(a, b)", "add(a: int, b: int) -> int")
    return {"draft": draft, "rounds": state["rounds"] + 1}

def critic(state: ReflectState) -> Command[Literal["generator", "__end__"]]:
    d = state["draft"]
    if "->" not in d:        fb = "add type hints"
    elif '"""' not in d:     fb = "add a docstring"
    else:                    fb = "APPROVED"
    print(f"  round {state['rounds']}: critic says {fb!r}")
    if fb == "APPROVED" or state["rounds"] >= 4:     # quality stop + safety stop
        return Command(goto=END, update={"feedback": [fb]})
    return Command(goto="generator", update={"feedback": [fb]})

b = StateGraph(ReflectState)
b.add_node("generator", generator); b.add_node("critic", critic)
b.add_edge(START, "generator"); b.add_edge("generator", "critic")
out = b.compile().invoke({"draft": "", "feedback": [], "rounds": 0})
print(out["draft"])

  round 1: critic says 'add type hints'
  round 2: critic says 'add a docstring'
  round 3: critic says 'APPROVED'
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a+b


**Which pattern should I pick?** Start with **one agent with good tools** (section 8). Add structure only when you can name the problem it solves:

| Problem you see | Pattern that helps |
|---|---|
| One prompt is trying to be an expert at everything | supervisor with specialists |
| Conversations move between departments | handoffs / swarm |
| Too many workers for one supervisor to manage | hierarchical teams |
| Long tasks wander off track | plan-and-execute |
| Outputs are often almost right | reflection |
| Independent sub-tasks are slow when done one by one | Send API fan-out (section 14) |

The companion notebook [13_crewai_and_multi_agent_frameworks_hands_on](13_crewai_and_multi_agent_frameworks_hands_on.ipynb) covers the failure modes of multi-agent systems and when **not** to use them.

## 18. A quick look at the Functional API (`@entrypoint` and `@task`)

**What it is.** Besides `StateGraph`, LangGraph has a second way to write workflows: plain Python functions with two decorators. `@task` marks a unit of work (its result is checkpointed and can run in parallel), and `@entrypoint` marks the main function. You get checkpoints, interrupts and streaming **without** drawing a graph.

**When to use which.** Use `StateGraph` when you want an explicit, drawable structure (most agents). Use the Functional API to add persistence to existing straight-line Python code.

In [28]:
from langgraph.func import entrypoint, task

@task
def score_essay(essay: str) -> int:
    return len(essay.split())                 # stand-in for an LLM grader

@entrypoint(checkpointer=InMemorySaver())
def grade_all(essays: list[str]) -> dict:
    futures = [score_essay(e) for e in essays]     # tasks start in parallel
    scores = [f.result() for f in futures]
    return {"scores": scores, "best": max(scores)}

print(grade_all.invoke(["short one", "a somewhat longer essay here", "mid size essay"],
                       {"configurable": {"thread_id": "essays"}}))

{'scores': [2, 5, 3], 'best': 5}


## 19. Reliability: error handling, retries and `recursion_limit`

Real LLM apps fail in boring ways: network timeouts, rate limits, malformed tool arguments, infinite loops. LangGraph gives you three tools:

| Tool | What it does | Use for |
|---|---|---|
| **try/except inside a node** + an `error` key in the state + a fallback route | turns a crash into a normal, routable state | bad tool output, parsing errors, "use a backup source" |
| **`RetryPolicy`** on a node | re-runs a failing node with exponential backoff (wait a bit longer each time) | **transient** errors: timeouts, HTTP 429/5xx |
| **`recursion_limit`** in the config | stops a run after N super-steps with `GraphRecursionError`. The default has changed between versions (it was 25 for a long time, some newer versions use a much larger value), so **set it yourself** | runaway loops |

**Analogy.** A delivery driver: if the door is locked, leave a note and go to the neighbour (fallback). If the road is briefly blocked, wait and try again (retry). If you have driven in circles for an hour, stop and call the office (recursion limit).

Retries are for errors that may go away by themselves. Do **not** retry a bug or a bad prompt; route it to a fallback or a human instead.

In [29]:
attempts = {"n": 0}
def flaky_api(state: Calc) -> dict:
    attempts["n"] += 1
    if attempts["n"] < 3:
        raise ConnectionError(f"timeout on attempt {attempts['n']}")    # simulated network blip
    return {"log": [f"API ok on attempt {attempts['n']}"]}

b = StateGraph(Calc)
b.add_node("call_api", flaky_api,
           retry_policy=RetryPolicy(max_attempts=4, initial_interval=0.01, backoff_factor=2.0,
                                    jitter=False, retry_on=ConnectionError))
b.add_edge(START, "call_api")
print(b.compile().invoke({"n": 0, "log": []}))

{'n': 0, 'log': ['API ok on attempt 3']}


In [30]:
class Lookup(TypedDict):
    query: str
    result: str
    error: str

def primary_source(state: Lookup) -> dict:
    try:
        raise KeyError("primary database has no entry")      # simulated failure
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}         # record it instead of crashing

def route_on_error(state: Lookup) -> str:
    return "fallback" if state.get("error") else END

b = StateGraph(Lookup)
b.add_node("primary", primary_source)
b.add_node("fallback", lambda s: {"result": f"cached answer for {s['query']!r}"})
b.add_edge(START, "primary")
b.add_conditional_edges("primary", route_on_error, {"fallback": "fallback", END: END})
print(b.compile().invoke({"query": "opening hours"}))

{'query': 'opening hours', 'result': "cached answer for 'opening hours'", 'error': "KeyError: 'primary database has no entry'"}


In [31]:
from langgraph.errors import GraphRecursionError

b = StateGraph(Calc)
b.add_node("forever", lambda s: {"n": s["n"] + 1})
b.add_edge(START, "forever"); b.add_edge("forever", "forever")      # a loop with NO stop condition
runaway = b.compile()
try:
    runaway.invoke({"n": 0, "log": []}, {"recursion_limit": 10})
except GraphRecursionError as e:
    print("Stopped safely:", str(e).split(".")[0])

Stopped safely: Recursion limit of 10 reached without hitting a stop condition


## 20. Streaming tokens to a UI (concept + tiny demo)

**What it is.** Users do not like staring at a spinner. With `stream_mode="messages"`, LangGraph forwards each **token** (small piece of text) as soon as the LLM inside a node produces it, together with metadata such as which node it came from.

**How it works in a web app.**

```mermaid
flowchart LR
    U[Browser chat UI] -- "POST /chat" --> API[Web server]
    API -- "graph.stream(..., stream_mode='messages')" --> G[LangGraph]
    G -- tokens --> API
    API -- "Server-Sent Events / WebSocket" --> U
```

**SSE** (Server-Sent Events) is a simple way for a server to keep pushing text to a browser over one HTTP connection. A sketch with FastAPI (not run here):

```python
# pip install fastapi uvicorn   (sketch only)
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
app = FastAPI()

@app.get("/chat")
def chat(q: str, thread: str):
    def events():
        cfg = {"configurable": {"thread_id": thread}}
        for token, meta in graph.stream({"messages": [("user", q)]}, cfg, stream_mode="messages"):
            if meta["langgraph_node"] == "agent" and token.content:
                yield f"data: {token.content}\n\n"          # one SSE event per token
    return StreamingResponse(events(), media_type="text/event-stream")
```

Below, the fake model splits its reply into word-sized chunks, just like a real streaming model would.

In [32]:
streaming_llm = GenericFakeChatModel(messages=iter(["LangGraph streams one token at a time to your UI."]))

def talker(state: MessagesState) -> dict:
    return {"messages": [streaming_llm.invoke(state["messages"])]}

b = StateGraph(MessagesState); b.add_node("agent", talker); b.add_edge(START, "agent")
ui_graph = b.compile()

chunks = []
for token, meta in ui_graph.stream({"messages": [HumanMessage("hi")]}, stream_mode="messages"):
    chunks.append(token.content)          # a real UI would send each chunk to the browser now
print("chunks as they arrived:", chunks)
print("what the user sees    :", "".join(chunks))
print("chunks came from node :", meta["langgraph_node"])

chunks as they arrived: ['LangGraph', ' ', 'streams', ' ', 'one', ' ', 'token', ' ', 'at', ' ', 'a', ' ', 'time', ' ', 'to', ' ', 'your', ' ', 'UI.']
what the user sees    : LangGraph streams one token at a time to your UI.
chunks came from node : agent


## 21. Deploying and observing a LangGraph app

**Deployment options (described generally; product names and plans change, check the docs as of late 2026):**

| Option | What you do | Good for |
|---|---|---|
| **Your own server** | wrap `graph.invoke/stream` in FastAPI (section 20), use a Postgres checkpointer | full control, existing infrastructure |
| **LangGraph Server / Platform** (from LangChain; recently also branded "LangSmith Deployment") | describe your graph in a `langgraph.json` file; the server gives you a ready HTTP API with threads, runs, background jobs, streaming, cron jobs and HITL endpoints | teams that want agent infrastructure without building it |
| **Local dev server** | `langgraph dev` (from `langgraph-cli`) runs the same API locally with a visual debugger ("LangGraph Studio") | development and demos |

A minimal `langgraph.json` looks roughly like this:

```json
{
  "dependencies": ["."],
  "graphs": { "support_agent": "./my_app/agent.py:graph" },
  "env": ".env"
}
```

**Production checklist (short)**
- Persistent checkpointer (Postgres) so runs survive restarts ("durable execution").
- `thread_id` per conversation; a Store for per-user long-term memory.
- `recursion_limit`, `RetryPolicy`, timeouts, and a cost budget per run.
- HITL (`interrupt`) before risky actions.
- Tracing and evaluations (below and [Agent observability](06_agent_observability.ipynb)).
- Security: least-privilege tools, input/output checks ([Agent security](08_agent_security.ipynb)).

**LangSmith tracing.** **LangSmith** is LangChain's tracing and evaluation service. A **trace** is the recorded tree of every node, LLM call and tool call in one run, with inputs, outputs, timings and token counts. LangGraph sends traces automatically when these environment variables are set (no code changes):

```bash
export LANGSMITH_TRACING=true
export LANGSMITH_API_KEY=...        # from your LangSmith account
export LANGSMITH_PROJECT=my-agent   # optional: group traces by project
```

The cell below only checks whether tracing is configured; it never calls the internet unless you set a key yourself. Open-source alternatives (Langfuse, Arize Phoenix, OpenTelemetry-based tools) are covered in [Agent observability](06_agent_observability.ipynb).

In [33]:
if os.environ.get("LANGSMITH_API_KEY") and os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    print("LangSmith tracing is ON: runs in this notebook will appear in project",
          os.environ.get("LANGSMITH_PROJECT", "default"))
else:
    print("skipped: LangSmith tracing needs LANGSMITH_TRACING=true and LANGSMITH_API_KEY (not set, running offline)")

skipped: LangSmith tracing needs LANGSMITH_TRACING=true and LANGSMITH_API_KEY (not set, running offline)


## 22. Putting it together: a small "production-style" support agent

One graph that combines most of this notebook: tool calls with `ToolNode`, an approval step with `interrupt()` before a risky tool, a checkpointer with `thread_id`, retries on the tool node, and a `recursion_limit`. The LLM is again the scripted fake.

In [34]:
@tool
def lookup_order(order_id: str) -> str:
    """Look up an order's status."""
    return {"A-1": "delivered, $40", "A-2": "delivered, $900"}.get(order_id, "not found")

@tool
def issue_refund(order_id: str, amount: float) -> str:
    """Refund an order. RISKY: needs human approval."""
    return f"refunded ${amount} on {order_id}"

RISKY = {"issue_refund"}
support_llm = ScriptedChatModel(messages=iter([
    AIMessage("", tool_calls=[{"name": "lookup_order", "args": {"order_id": "A-2"}, "id": "t1"}]),
    AIMessage("", tool_calls=[{"name": "issue_refund", "args": {"order_id": "A-2", "amount": 900}, "id": "t2"}]),
    AIMessage("Done: your refund of $900 for order A-2 is on its way."),
]))

def support_agent(state: MessagesState) -> dict:
    return {"messages": [support_llm.invoke(state["messages"])]}

def human_gate(state: MessagesState) -> Command[Literal["tools", "agent"]]:
    calls = state["messages"][-1].tool_calls
    risky = [c for c in calls if c["name"] in RISKY]
    if risky:
        ok = interrupt({"approve?": [(c["name"], c["args"]) for c in risky]})
        if ok != "yes":   # rejected: tell the LLM instead of running the tool
            msgs = [ToolMessage("Rejected by a human reviewer.", tool_call_id=c["id"]) for c in calls]
            return Command(goto="agent", update={"messages": msgs})
    return Command(goto="tools")

b = StateGraph(MessagesState)
b.add_node("agent", support_agent)
b.add_node("human_gate", human_gate)
b.add_node("tools", ToolNode([lookup_order, issue_refund]),
           retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.01))
b.add_edge(START, "agent")
b.add_conditional_edges("agent", tools_condition, {"tools": "human_gate", END: END})
b.add_edge("tools", "agent")
support = b.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "ticket-900"}, "recursion_limit": 15}
first = support.invoke({"messages": [HumanMessage("Please refund order A-2")]}, cfg)
print("PAUSED for approval:", first["__interrupt__"][0].value)
final = support.invoke(Command(resume="yes"), cfg)
show(final["messages"])

PAUSED for approval: {'approve?': [('issue_refund', {'order_id': 'A-2', 'amount': 900})]}


  HumanMessage  'Please refund order A-2'
  AIMessage     ''  tool_calls=[('lookup_order', {'order_id': 'A-2'})]
  ToolMessage   'delivered, $900'
  AIMessage     ''  tool_calls=[('issue_refund', {'order_id': 'A-2', 'amount': 900})]
  ToolMessage   'refunded $900.0 on A-2'
  AIMessage     'Done: your refund of $900 for order A-2 is on its way.'


## 23. Recap / cheat sheet

| Concept | One-line meaning | API |
|---|---|---|
| State | shared dict all nodes read and update | `class S(TypedDict)`, `MessagesState` |
| Reducer | how an update merges into a key | `Annotated[list, operator.add]`, `add_messages` |
| Node | a function `state -> partial update` | `builder.add_node("name", fn)` |
| Edge | fixed "what runs next" | `add_edge(a, b)`, `START`, `END` |
| Conditional edge | routing function picks next node | `add_conditional_edges(src, fn, path_map)` |
| Cycle | edge back to an earlier node; needs a stop rule | conditional edge + max tries |
| Compile / run | validate, then execute | `compile()`, `invoke()`, `stream(stream_mode=...)` |
| Draw | Mermaid text of the graph | `graph.get_graph().draw_mermaid()` |
| Tool agent | LLM + ToolNode loop | `ToolNode`, `tools_condition`, `create_react_agent` |
| Checkpointer | snapshot after every step, per thread | `InMemorySaver`, `thread_id` |
| Time travel | look back, replay, fork | `get_state_history`, `invoke(None, cfg)`, `update_state` |
| HITL | pause for a human, then resume | `interrupt()`, `Command(resume=...)` |
| Command | update + jump in one return | `Command(update=..., goto=...)` |
| Send | dynamic parallel fan-out | `[Send("node", payload), ...]` |
| Subgraph | compiled graph used as a node | `add_node("team", compiled_child)` |
| Store | long-term memory across threads | `InMemoryStore`, `put/get/search` |
| Reliability | survive failures and loops | `RetryPolicy`, try/except + fallback, `recursion_limit` |
| Streaming tokens | typing effect in UIs | `stream_mode="messages"` |

## 24. Interview questions

1. **Why use LangGraph instead of a plain `while` loop agent?** The control flow becomes explicit (named nodes and edges), so it is easier to debug, test and draw, and you get checkpoints, pause/resume, time travel and streaming for free.
2. **What is a reducer and when do you need one?** A function that merges a node's update into the existing value. You need one whenever a key should accumulate (messages, logs) or when parallel nodes write the same key; otherwise LangGraph raises `InvalidUpdateError`.
3. **What is the difference between a checkpointer and a Store?** A checkpointer saves the full state of **one thread** after every step (short-term memory, resume, HITL). A Store holds small documents shared **across threads** (long-term memory such as user preferences).
4. **How does `interrupt()` work, and what is the common pitfall?** It saves state and returns control to the caller; `Command(resume=value)` continues and makes `interrupt` return `value`. Pitfall: the node re-runs from the top on resume, so side effects before the interrupt happen twice.
5. **When would you return `Command` from a node instead of using a conditional edge?** When the node itself computes the decision and also updates state (for example agent handoffs), or to jump into the parent graph with `Command.PARENT`.
6. **What problem does the Send API solve?** Fan-out where the number of parallel tasks is only known at run time (map-reduce over documents or sub-questions); results are merged with a reducer.
7. **Supervisor vs swarm?** Supervisor: one central router decides every step, easy to control and audit. Swarm/handoffs: agents pass control directly to peers, fewer LLM calls and natural for conversations, but harder to reason about globally.
8. **How do you stop an agent looping forever?** A stop condition in the routing logic (quality and max-iterations), plus `recursion_limit` as a global safety net, plus cost/time budgets.
9. **Retry vs fallback?** Retry (`RetryPolicy`) for transient errors that may succeed next time; fallback route (error in state) for errors that will not fix themselves.
10. **How would you stream tokens to a web UI?** `graph.stream(..., stream_mode="messages")`, filter by `metadata["langgraph_node"]`, and push each chunk to the browser via SSE or WebSockets.

## 25. Further reading

- LangGraph documentation (concepts: state, reducers, persistence, human-in-the-loop, streaming, multi-agent) - https://langchain-ai.github.io/langgraph/ and the LangChain docs site (docs.langchain.com), which now hosts the LangGraph v1 docs.
- LangChain blog, "LangGraph: Multi-Agent Workflows" (2024), and the `langgraph-supervisor` / `langgraph-swarm` repositories on GitHub.
- Yao et al., "ReAct: Synergizing Reasoning and Acting in Language Models" (2022), arXiv:2210.03629.
- Wang et al., "Plan-and-Solve Prompting" (2023), and the LangGraph "plan-and-execute" tutorial.
- Shinn et al., "Reflexion: Language Agents with Verbal Reinforcement Learning" (2023), arXiv:2303.11366.
- Madaan et al., "Self-Refine: Iterative Refinement with Self-Feedback" (2023), arXiv:2303.17651.
- Anthropic, "Building effective agents" (December 2024): workflows vs agents, and why to start simple.
- LangSmith documentation (tracing and evaluation) - https://docs.smith.langchain.com/
- In this repo: [03_langgraph](03_langgraph.ipynb), [06_agent_observability](06_agent_observability.ipynb), [10_harness_engineering_and_agentic_engineering](10_harness_engineering_and_agentic_engineering.ipynb), [11_context_engineering_memory_and_agent_protocols](11_context_engineering_memory_and_agent_protocols.ipynb), [13_crewai_and_multi_agent_frameworks_hands_on](13_crewai_and_multi_agent_frameworks_hands_on.ipynb).